In [ ]:
!pip install smolagents python-dotenv sqlalchemy --upgrade -q

In [ ]:
from google.colab import userdata


my_token = userdata.get('HF_TOKEN')
with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={my_token}")


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [ ]:
import json

question = None
with open("law_episode.json", "r") as f:
    questions = json.load(f)

with open("golds.json", 'r') as v:
  golds = json.load(v)
  q_ids = {g["question_id"]: g for g in golds}

In [ ]:
from sqlalchemy import create_engine, inspect, text

db_path = "sqlite:///law_episode.sqlite"
db_name = "law_episode.sqlite"

if not os.path.exists("law_episode.sqlite"):
    print("WARNING: db not found.")

engine = create_engine(db_path)

In [ ]:
inspector = inspect(engine)
table_names = inspector.get_table_names()

schema = "Database Schema:\n"

for table in table_names:
    schema += f"Table: {table}\n"
    columns = inspector.get_columns(table)
    for col in columns:
        schema += f"  - {col['name']} ({col['type']})\n"

print(schema)

Database Schema:
Table: Award
  - award_id (INTEGER)
  - organization (TEXT)
  - year (INTEGER)
  - award_category (TEXT)
  - award (TEXT)
  - series (TEXT)
  - episode_id (TEXT)
  - person_id (TEXT)
  - role (TEXT)
  - result (TEXT)
Table: Credit
  - episode_id (TEXT)
  - person_id (TEXT)
  - category (TEXT)
  - role (TEXT)
  - credited (TEXT)
Table: Episode
  - episode_id (TEXT)
  - series (TEXT)
  - season (INTEGER)
  - episode (INTEGER)
  - number_in_series (INTEGER)
  - title (TEXT)
  - summary (TEXT)
  - air_date (DATE)
  - episode_image (TEXT)
  - rating (REAL)
  - votes (INTEGER)
Table: Keyword
  - episode_id (TEXT)
  - keyword (TEXT)
Table: Person
  - person_id (TEXT)
  - name (TEXT)
  - birthdate (DATE)
  - birth_name (TEXT)
  - birth_place (TEXT)
  - birth_region (TEXT)
  - birth_country (TEXT)
  - height_meters (REAL)
  - nickname (TEXT)
Table: Vote
  - episode_id (TEXT)
  - stars (INTEGER)
  - votes (INTEGER)
  - percent (REAL)



In [ ]:
from smolagents import tool
@tool
def sql_engine(query: str) -> str:
    """
    Allows you to perform SQL queries on the table. Returns a string representation of the result.

    Args:
        query: The query to perform.
    """
    output = ""
    MAX_ROWS = 10
    with engine.connect() as con:
        rows = con.execute(text(query))
        for i, row in enumerate(rows):
          if i > MAX_ROWS:
            output += "\n...Output truncated."
            return output
          else:
            output += "\n" + str(row)

    return output

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, EMPTY_PROMPT_TEMPLATES

# Define the optimized system prompt combining PDF Examples + Smolagents Technical Logic
system_prompt = """
You are an expert Data Scientist specialized in Text-to-SQL tasks. Your goal is to answer natural language questions by generating valid, executable SQL queries.

You will be given a task to solve as best you can.
To do so, you have been given access to the tool 'sql_engine': this tool is basically a Python function which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

PROTOCOL:
1. Reasoning Trace: Explicitly state your plan in the 'Thought' section.
2. Schema Understanding: Use the provided database schema to understand:
   - table names
   - column names
   - primary/foreign key relationships
   - bridge tables
3. Test Your Logic: Do not generate the final answer yet.
   - You must run a "Test Query" to verify your hypothesis.
   - Check if your filters exist (e.g., `WHERE Name LIKE '%Apple%'` vs `WHERE Name = 'Apple'`).
   - Check if your JOINs return rows.
   - If the test returns 0 rows or an error, you must revise your query and test again.
4. After every exploratory or validation query executed through sql_engine, you MUST:
   - Read the result.
   - Produce a new explicit `Thought:` section interpreting the output.
   - Decide the next step based on that interpretation before running another code block.

CRITICAL RULES (YOU MUST FOLLOW):
1. Python Syntax Only: The code block contains PYTHON code.
2. SQL as Strings: Always pass SQL as a string to `sql_engine`: `print(sql_engine(query="SELECT ..."))`
3. HANDLING JOINS (NO SHORTCUTS):
    - Check Foreign Keys: Do NOT join on `id` unless you are sure it is a foreign key.
    - Bridge Tables: If Table A and Table C have no direct link, look for a middle table B:
      Example: Works -> Chapters -> Paragraphs -> Characters. You cannot jump from Works to Characters.
4. NO PYTHON DATA MANIPULATION: Do not fetch data into Python to filter it. Write a single SQL query using JOINs or Subqueries.
5. FINAL ANSWER FORMAT: NO BLIND SUBMISSIONS:
   - You must see the output of a successful Test Query in the logs before you are allowed to submit.
   - Your final goal is the SQL QUERY STRING, not the data result.
   - DO NOT EXECUTE the final query inside `final_answer`:
    * WRONG: `final_answer(sql_engine(query="SELECT..."))` <-- returns data (BAD)
    * CORRECT: `final_answer("SELECT...")` <-- returns SQL (GOOD)

WORKED EXAMPLE TRACE:

Task: "Return the names of companies in the ‘AI’ sector that were founded before 2015 and have at least two offices in France."

Thought: I need to find companies meeting three criteria: sector 'AI', founded before 2015, and having > 1 office in France. First, I need to inspect the tables to understand the schema. I will use Python to call the sql_engine.
<code>
# Inspect tables using Python strings for the queries
print(sql_engine(query="SELECT * FROM Company LIMIT 3"))
print(sql_engine(query="SELECT * FROM Office LIMIT 3"))
</code>

[ANS]
cid | name | sector | founded_year
1 | EURECOM | Education | 1990
2 | Acme AI | AI | 2012
3 | RoboCorp | AI | 2016

oid | cid | country | city
10 | 2 | France | Paris
11 | 2 | France | Lyon
12 | 2 | Germany | Berlin
[/ANS]

Thought: The tables are 'Company' and 'Office', joined by 'cid'.
- 'founded_year' is an integer.
- 'sector' is a string.
Now I will construct a validation query. I must assign the SQL to a Python variable as a string.

<code>
# Validating the logic with a count. Notice the query is a string.
validation_query = \"\"\"
SELECT c.cid, c.name, COUNT(*) AS fr_offices
FROM Company c
JOIN Office o ON o.cid=c.cid
WHERE c.sector='AI' AND c.founded_year<2015 AND o.country='France'
GROUP BY c.cid, c.name
HAVING COUNT(*)>=2
LIMIT 3
\"\"\"
print(sql_engine(query=validation_query))
</code>

[ANS]
cid | name | fr_offices
2 | Acme AI | 2
[/ANS]

Thought: The logic is correct. Acme AI is identified. I can now generate the final SQL query string.
<code>
final_sql = "\"\"
SELECT c.name
FROM Company AS c
JOIN Office AS o ON o.cid = c.cid
WHERE c.sector = 'AI'
AND c.founded_year < 2015
AND o.country = 'France'
GROUP BY c.cid, c.name
HAVING COUNT(*) >= 2
"\"\"
final_answer(final_sql)
</code>

Other example:
2. Task: How many scenes are there in Act 1 in Twelfth Night?

Thought: I analyze first the chapters table, to get a better understanding of its format.

<code>
test_query = "\"\"
SELECT *
from chapters
LIMIT 3
"\"\"
print(sql_engine(test_query))
</code>

[ANS]
id | Act | Scene | Description | work_id
18704 | 1 | 1 | DUKE ORSINO’s palace. | 1
18705 | 1 | 2 | The sea-coast. | 1
18706 | 1 | 3 | OLIVIA’S house. | 1
[/ANS]

Thought: I can use the work_id foreign key to perform a join between the works and chapters table. I need to consider only the work 'Twelfth Night', so a filter is needed.

<code>
test_query = "\"\"
SELECT w.Title, c.Act, c.Scene
FROM works w JOIN chapters c ON c.work_id = w.id
WHERE w.Title = 'Twelfth Night'
LIMIT 3
"\"\"
print(sql_engine(query))
</code>

[ANS]
Title | Act | Scene
Twelfth Night | 1 | 1
Twelfth Night | 1 | 2
Twelfth Night | 1 | 3
[/ANS]

Thought: As expected, I have now the information about all the acts and the scenes of the work 'Twelfth Night'. Since I need to count the number of scenes in Act 1, a further filter is needed.

<code>
test_query = "\"\"
SELECT w.Title, c.Act, c.Scene
FROM works w JOIN chapters c ON c.work_id = w.id
WHERE w.Title = 'Twelfth Night' AND c.Act = 1
LIMIT 3
"\"\"
print(sql_engine(query))
</code>

[ANS]
Title | Act | Scene
Twelfth Night | 1 | 1
Twelfth Night | 1 | 2
Twelfth Night | 1 | 3
[/ANS]

Thought: Now that I have only the instances related to Act 1, I can proceed with the final query in which the number of scenes of Act 1 in the work 'Twelfth Night' are counted. The column is renamed for better understanding.

<code>
test_query = "\"\"
SELECT COUNT(*) as n_Scenes
FROM works w JOIN chapters c ON c.work_id = w.id
WHERE w.Title = 'Twelfth Night' AND c.Act = 1
"\"\"
print(sql_engine(test_query))
</code>

[ANS]
n_Scenes
5
[/ANS]

Thought: The number of scenes in Act 1 in Twelfth Night is correctly retrieved, I can proceed with returning the final query.

<code>
final_sql = "\"\"
SELECT COUNT(*) as n_Scenes
FROM works w JOIN chapters c ON c.work_id = w.id
WHERE w.Title = 'Twelfth Night' AND c.Act = 1
"\"\"
final_answer(final_sql)
</code>

---

You only have access to these tools, behaving like regular python functions:
{{code_block_opening_tag}}
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{% endfor %}
{{code_block_closing_tag}}

Here are the rules you should always follow to solve your task:
1. Always provide a 'Thought:' sequence, and a '{{code_block_opening_tag}}' sequence ending with '{{code_block_closing_tag}}', else you will fail.
2. Use only variables that you have defined!
3. Always use the right arguments for the tools. DO NOT pass the arguments as a dict as in 'answer = wikipedia_search({'query': \"What is the place where James Bond lives?\"})', but use the arguments directly as in 'answer = wikipedia_search(query=\"What is the place where James Bond lives?\")'.
4. For tools WITHOUT JSON output schema: Take care to not chain too many sequential tool calls in the same code block, as their output format is unpredictable. For instance, a call to wikipedia_search without a JSON output schema has an unpredictable return format, so do not have another tool call that depends on its output in the same block: rather output results with print() to use them in the next block.
5. For tools WITH JSON output schema: You can confidently chain multiple tool calls and directly access structured output fields in the same code block! When a tool has a JSON output schema, you know exactly what fields and data types to expect, allowing you to write robust code that directly accesses the structured response (e.g., result['field_name']) without needing intermediate print() statements.
6. Call a tool only when needed, and never re-do a tool call that you previously did with the exact same parameters.
7. Don't name any new variable with the same name as a tool: for instance don't name a variable 'final_answer'.
8. Never create any notional variables in our code, as having these in your logs will derail you from the true variables.
9. You can use imports in your code, but only from the following list of modules: {{authorized_imports}}
10. The state persists between code executions: so if in one step you've created variables or imported modules, these will all persist.
11. Don't give up! You're in charge of solving the task, not providing directions to solve it.

{%- if custom_instructions %}
{{custom_instructions}}
{%- endif %}

Now Begin!
"""


my_templates_dict = EMPTY_PROMPT_TEMPLATES.copy()
my_templates_dict["system_prompt"] = system_prompt

In [ ]:
agent = CodeAgent(
    tools=[sql_engine],
    model=InferenceClientModel(model_id="Qwen/Qwen3-8B", token=my_token),
    prompt_templates=my_templates_dict,
    verbosity_level=2
)

In [ ]:
import re

def get_stats(agent):


    log_parts = []
    sql_query = None
    reasoning_len = 0
    tool_call_count = 0
    errors_count = 0
    is_final_answer = False

    for i, step in enumerate(agent.memory.steps):

        if i == 0: continue # Skip the task step

        if step.is_final_answer:
          sql_query = step.action_output.strip()
          is_final_answer = True

        thought = getattr(step, 'model_output', getattr(step, 'thought', None))
        if thought:
            print(thought)
            clean_thought = re.sub(r'<code>.*?</code>', '', thought, flags=re.DOTALL)
            clean_thought = clean_thought.replace('\n', ' ').strip()
            if clean_thought:
                log_parts.append(f"{clean_thought}")

        # call
        if hasattr(step, 'tool_calls') and step.tool_calls and not is_final_answer:
            for tool_call in step.tool_calls:
              if step.is_final_answer:
                continue
              tool_call_count += 1
              args = getattr(tool_call, 'arguments', str(tool_call))

              match_triple = re.search(r'=\s*"""(.*?)"""', args, re.DOTALL)
              match_direct = re.search(r'sql_engine\s*\(\s*query\s*=\s*["\'](.*?)["\']', args, re.DOTALL)

              found_sql = None
              if match_triple:
                  found_sql = match_triple.group(1)
              elif match_direct:
                  found_sql = match_direct.group(1)

              if found_sql:
                  # Flatten SQL (remove newlines for single-line log)
                  flat_sql = found_sql.replace('\n', ' ').replace('\\n', ' ').replace("   ", " ").strip()
                  log_parts.append(f"[CALL] {flat_sql}")

        # ans - obs
        if hasattr(step, 'observations') and step.observations and not is_final_answer:

          obs = str(step.observations).strip()


          obs = obs.replace("Execution logs:", "").replace("Last output from code snippet:", "")
          obs = re.sub(r'\bNone\b', '', obs)
          obs = re.sub(r"^\('(.+)',\)$", r"\1", obs.strip(), flags=re.MULTILINE)
          obs = re.sub(r"^\('(.+)'\)$", r"\1", obs.strip(), flags=re.MULTILINE)

          obs_clean = obs.strip().replace('\n', ' ')

          if not obs_clean or re.fullmatch(r'[\[\]\(\)\s,]*', obs_clean):
              log_parts.append("[ANS] (no rows) [/ANS]")

          # Truncate if too long
          if len(obs_clean) > 200:
              obs_clean = obs_clean[:200] + "... [truncated]"

          if obs_clean:
              log_parts.append(f"[ANS] {obs_clean} [/ANS]")

        # errors
        if hasattr(step, 'error') and step.error:
             err_clean = str(step.error).replace('\n', ' ')
             errors_count += 1
             log_parts.append(f"[ERROR] {err_clean}")


    # Join everything with a single newline between steps
    full_log_string = " ".join(log_parts)
    full_log_string = full_log_string.replace("\\", "").replace("\"", "\'")
    reasoning_len = len(full_log_string)
    sql_query = sql_query.replace("\\n",  "").replace("\n", " ").replace("\\'", "'").strip()


    return full_log_string, sql_query, tool_call_count, errors_count, reasoning_len


In [ ]:
def compute_execution_accuracy(gt_results, predict_results):
  num_correct = 0
  num_queries = len(gt_results)
  mismatch_idx = []

  for i, result in enumerate(gt_results):
      if set(result['results']) == set(predict_results[i]['results']):
          num_correct += 1
      else:
          mismatch_idx.append(i)

  acc = (num_correct / num_queries)

  return acc

In [ ]:
import sqlite3
def run_query(db_path, query):
  conn = sqlite3.connect(db_path)
  try:
    cursor = conn.cursor()
    cursor.execute(query)
    rows = cursor.fetchall()
    conn.close()

    # Flatten results and convert to list of strings
    return [row[0] for row in rows], True
  except:
    return [], False

In [ ]:
import time

traces = []
i = 1
for q in questions:
  trace_accuracy = None
  question = q["questions"]
  evidence = q["evidence"]
  difficulty = q["difficulty"]
  q_id = q["question_id"]

  gt_query = q_ids[q_id]["target_sql"]

  USER_PROMPT = f"""DB Schema: {schema}. Question: {evidence}. {question}"""

  print(f"--- Question {i} ---")
  i += 1

  start_time = time.time()
  agent.run(USER_PROMPT)
  end_time = time.time()

  log_string, pred_query, tool_call_count, errors_count, reasoning_len = get_stats(agent)

  rows_gt, _ = run_query(db_name, gt_query)
  gt_res = [{"results": rows_gt}]

  rows_pred, is_valid_sql = run_query(db_name, pred_query)
  pred_res= [{"results": rows_pred}]

  if is_valid_sql:
    acc = compute_execution_accuracy(gt_res, pred_res)

  else:
    trace_accuracy = 0
    acc = 0


  complete_trace = {
    "question_id": q_id,
    "input": USER_PROMPT,
    "output": log_string,
    "difficulty": difficulty,
    "pred_query": pred_query,
    "target_query": gt_query,
    "tool_call_count": tool_call_count,
    "error_count": errors_count,
    "latency": round(end_time-start_time, 2),
    "reasoning_len": reasoning_len,
    "execution_accuracy": int(acc),
    "trace_accuracy": trace_accuracy
  }

  traces.append(complete_trace)


with open("law_episode_traces.json", "w") as f:
  json.dump(traces, f, indent=2, ensure_ascii=False)

--- Question 1 ---


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ DB Schema: Database Schema:                                                                                     │
│ Table: Award                                                                                                    │
│   - award_id (INTEGER)                                                                                          │
│   - organization (TEXT)                                                                                         │
│   - year (INTEGER)                                                                                              │
│   - award_category (TEXT)                                                                                       │
│   - award (TEXT)                                                                                                │
│   - series (TEXT)                                                                                               │
│   - episode_id (TEXT)                                                                                           │
│   - person_id (TEXT)                                                                                            │
│   - role (TEXT)                                                                                                 │
│   - result (TEXT)                                                                                               │
│ Table: Credit                                                                                                   │
│   - episode_id (TEXT)                                                                                           │
│   - person_id (TEXT)                                                                                            │
│   - category (TEXT)                                                                                             │
│   - role (TEXT)                                                                                                 │
│   - credited (TEXT)                                                                                             │
│ Table: Episode                                                                                                  │
│   - episode_id (TEXT)                                                                                           │
│   - series (TEXT)                                                                                               │
│   - season (INTEGER)                                                                                            │
│   - episode (INTEGER)                                                                                           │
│   - number_in_series (INTEGER)                                                                                  │
│   - title (TEXT)                                                                                                │
│   - summary (TEXT)                                                                                              │
│   - air_date (DATE)                                                                                             │
│   - episode_image (TEXT)                                                                                        │
│   - rating (REAL)                                                                                               │
│   - votes (INTEGER)                                                                                             │
│ Table: Keyword                                                                                                  │
│   - episode_id (TEXT)                                                                                           │
│   - keyword (TEXT)                                    

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                   
                                                                                                                   
Thought: I need to find all keywords associated with episodes that have a rating greater than 8. The relevant      
tables are Episode (for rating) and Keyword (for keywords). I will join these tables on episode_id and filter using
the rating condition.                                                                                              
                                                                                                                   
<code>                                                                                                             
# Validate the schema and join logic                                                                               
test_query = """                                                                                                   
SELECT e.episode_id, e.rating, k.keyword                                                                           
FROM Episode e                                                                                                     
JOIN Keyword k ON e.episode_id = k.episode_id                                                                      
WHERE e.rating > 8                                                                                                 
LIMIT 3                                                                                                            
"""                                                                                                                
print(sql_engine(query=test_query))                                                                                
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Validate the schema and join logic                                                                             
  test_query = """                                                                                                 
  SELECT e.episode_id, e.rating, k.keyword                                                                         
  FROM Episode e                                                                                                   
  JOIN Keyword k ON e.episode_id = k.episode_id                                                                    
  WHERE e.rating > 8                                                                                               
  LIMIT 3                                                                                                          
  """                                                                                                              
  print(sql_engine(query=test_query))                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                   
                                                                                                                   
The total number of stars for episodes in which Donna Villella worked is retrieved by joining the Credit and Vote  
tables, filtering by her person_id, and summing the stars. The result is 55.                                       
                                                                                                                   
<code>                                                                                                             
final_answer("SELECT SUM(v.stars) AS total_stars FROM Credit c JOIN Vote v ON c.episode_id = v.episode_id WHERE    
c.person_id = 'nm2650434'")                                                                                        
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("SELECT SUM(v.stars) AS total_stars FROM Credit c JOIN Vote v ON c.episode_id = v.episode_id WHERE  
  c.person_id = 'nm2650434'")                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: SELECT SUM(v.stars) AS total_stars FROM Credit c JOIN Vote v ON c.episode_id = v.episode_id WHERE 
c.person_id = 'nm2650434'

[Step 3: Duration 9.61 seconds| Input tokens: 8,593 | Output tokens: 3,342]



Thought: I need to find the total number of stars for episodes in which Donna Villella worked. First, I'll retrieve Donna Villella's person_id from the Person table. Then, I'll join with the Credit table to find the episodes she was involved in, and finally join with the Vote table to get the stars for those episodes. 

<code>
# Find Donna Villella's person_id
donna_person_id_query = """
SELECT person_id
FROM Person
WHERE name = 'Donna Villella'
LIMIT 1
"""
print(sql_engine(query=donna_person_id_query))
</code>


Thought: Now that I have Donna Villella's person_id as 'nm2650434', I need to find all episodes she worked on by joining the Credit table with the Vote table. Then, I can sum the stars from the Vote table for those episodes to get the total number of stars.

<code>
# Calculate total stars for episodes Donna Villella worked in
total_stars_query = """
SELECT SUM(v.stars) AS total_stars
FROM Credit c
JOIN Vote v ON c.episode_id = v.episode_id
WHERE c.person_id = 'nm2650434'
"""